In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

# Conectamos a la base de trabajo
con = duckdb.connect('../data/BaseDeDatos_Working.db')

# Inpeccion para el estandarizar los datos

## Lo que hare es que quitare en la los textos acentos extraños ademas poner en formato de Mayuscula y inpeccionar si hay valores con espacios invisibles 

In [ ]:
text_columns = con.execute("""
    SELECT table_name, column_name 
    FROM information_schema.columns 
    WHERE data_type = 'VARCHAR' 
      AND table_schema = 'main'
""").fetchall()

print(f"Auditando {len(text_columns)} columnas de texto...\n")

for table, col in text_columns:
    stats = con.execute(f"""
        SELECT 
            COUNT(*) as total,
            SUM(CASE WHEN "{col}" != TRIM("{col}") THEN 1 ELSE 0 END) as con_espacios,
            SUM(CASE WHEN regexp_matches("{col}", '[áéíóúÁÉÍÓÚüÜñÑ]') THEN 1 ELSE 0 END) as con_acentos_especiales,
            SUM(CASE WHEN "{col}" != UPPER("{col}") AND "{col}" != LOWER("{col}") THEN 1 ELSE 0 END) as mezcla_case
        FROM {table}
    """).fetchone()
    
    if stats[1] > 0 or stats[2] > 0 or stats[3] > 0:
        print(f"Tabla: {table} | Columna: {col}")
        print(f"   - Espacios extra (inicio/fin): {stats[1]}")
        print(f"   - Con acentos/caracteres: {stats[2]}")
        print(f"   - Mezcla Mayúsculas/Minúsculas: {stats[3]}")
        print("-" * 30)

Auditando 22 columnas de texto...

Tabla: causa | Columna: causa
   - Espacios extra (inicio/fin): 1
   - Con acentos/caracteres: 9
   - Mezcla Mayúsculas/Minúsculas: 54
------------------------------
Tabla: causa | Columna: causa_especifica
   - Espacios extra (inicio/fin): 0
   - Con acentos/caracteres: 10
   - Mezcla Mayúsculas/Minúsculas: 54
------------------------------
Tabla: danos | Columna: tamanio
   - Espacios extra (inicio/fin): 0
   - Con acentos/caracteres: 12703
   - Mezcla Mayúsculas/Minúsculas: 12703
------------------------------
Tabla: diccionario | Columna: fuente
   - Espacios extra (inicio/fin): 0
   - Con acentos/caracteres: 0
   - Mezcla Mayúsculas/Minúsculas: 3
------------------------------
Tabla: diccionario | Columna: nombre_completo
   - Espacios extra (inicio/fin): 0
   - Con acentos/caracteres: 0
   - Mezcla Mayúsculas/Minúsculas: 146
------------------------------
Tabla: diccionario | Columna: unidad_de_medida
   - Espacios extra (inicio/fin): 0
   - Con

In [3]:
# Ejemplo para la tabla municipios
df_variaciones = con.execute("""
    SELECT 
        LOWER(TRIM(nombre_municipio)) as version_estandar,
        ARRAY_AGG(DISTINCT nombre_municipio) as variaciones_encontradas,
        COUNT(DISTINCT nombre_municipio) as total_variaciones
    FROM municipios
    GROUP BY 1
    HAVING total_variaciones > 1
""").df()

print("Municipios con múltiples representaciones textuales:")
print(df_variaciones)

Municipios con múltiples representaciones textuales:
      version_estandar                     variaciones_encontradas  \
0       villa guerrero            [Villa GUERRERO, Villa Guerrero]   
1  san josé del rincón  [San José Del Rincón, San José del Rincón]   

   total_variaciones  
0                  2  
1                  2  
